In [56]:
import os
import json

def create_prediction_map(predictions_dict):
    """
    Converts the list-based prediction structure into a dictionary
    keyed by filename for efficient lookups.
    """
    prediction_map = {}
    # Process correct predictions
    for pred in predictions_dict.get("correct_predictions", []):
        filename = pred["filename"]
        # Remove the .png suffix from each neighbor filename
        neighbor_filenames_no_ext = [
            neighbor_filename.removesuffix(".png")
            for neighbor_filename in pred.get("top_k_neighbor_filenames", [])
        ]
        
        prediction_map[filename] = {
            "correct": True,
            "actual_label": pred["actual_label"],
            "predicted_label": pred["predicted_label"],
            "top_k_neighbor_filenames": neighbor_filenames_no_ext # Use the list without extensions
        }

    # Process incorrect predictions
    for pred in predictions_dict.get("incorrect_predictions", []):
        filename = pred["filename"]
        # Remove the .png suffix from each neighbor filename
        neighbor_filenames_no_ext = [
            neighbor_filename.removesuffix(".png")
            for neighbor_filename in pred.get("top_k_neighbor_filenames", [])
        ]
        
        prediction_map[filename] = {
            "correct": False,
            "actual_label": pred["actual_label"],
            "predicted_label": pred["predicted_label"],
            "top_k_neighbor_filenames": neighbor_filenames_no_ext # Use the list without extensions
    }
    # A small check to ensure we processed something
    if not prediction_map:
        print("Warning: Created an empty prediction map. Check the input dictionary.")
    return prediction_map

In [57]:
def analyze_and_print_comparison(dataset_name, base_predictions_path, finetuned_predictions_path, valid_full_filenames_set=None):
    """
    Loads, compares, and prints a summary for a given pair of base and finetuned
    prediction files.
    """
    print("\n" + "#"*60)
    print(f"### Analyzing Dataset: {dataset_name}")
    print("#"*60 + "\n")

    # --- 1. Load Data ---
    base_predictions = {}
    finetuned_predictions = {}

    if os.path.exists(base_predictions_path):
        with open(base_predictions_path, 'r') as f:
            base_predictions = json.load(f)
        print(f"Successfully loaded base predictions from {base_predictions_path}")
    else:
        print(f"ERROR: Base predictions file not found at {base_predictions_path}")
        return # Exit this analysis if a file is missing

    if os.path.exists(finetuned_predictions_path):
        with open(finetuned_predictions_path, 'r') as f:
            finetuned_predictions = json.load(f)
        print(f"Successfully loaded finetuned predictions from {finetuned_predictions_path}")
    else:
        print(f"ERROR: Finetuned predictions file not found at {finetuned_predictions_path}")
        return # Exit this analysis if a file is missing

    # --- 2. Create Prediction Maps ---
    base_map = create_prediction_map(base_predictions)
    finetuned_map = create_prediction_map(finetuned_predictions)
    print(f"Created base map with {len(base_map)} entries and finetuned map with {len(finetuned_map)} entries.")

    valid_basenames_set = None
    if valid_full_filenames_set is not None:
        # os.path.splitext('file.png') returns ('file', '.png')
        valid_basenames_set = {os.path.splitext(f)[0] for f in valid_full_filenames_set}
        print(f"Filtering enabled. Query images and ALL neighbors must be in the valid set of {len(valid_full_filenames_set)} files.")

    # --- 3. Find Intersections and Categorize ---
    both_correct = []
    both_incorrect = []
    finetuned_fixed = []
    finetuned_broke = []

    common_filenames = set(base_map.keys()) & set(finetuned_map.keys())
    if not common_filenames:
        print("ERROR: No common filenames found between the two prediction sets. Cannot compare.")
        return
    

    for filename in common_filenames:
        base_pred = base_map[filename]
        ft_pred = finetuned_map[filename]

        if valid_full_filenames_set is not None:
            # Condition 1: The query image itself must be valid.
            if filename not in valid_full_filenames_set:
                continue

            # Condition 2: ALL neighbors for the base prediction must be valid.
            # We use all() to ensure every neighbor is in the set of valid basenames.
            if not all(neighbor in valid_basenames_set for neighbor in base_pred["top_k_neighbor_filenames"]):
                continue

            # Condition 3: ALL neighbors for the finetuned prediction must be valid.
            if not all(neighbor in valid_basenames_set for neighbor in ft_pred["top_k_neighbor_filenames"]):
                continue



        if base_pred["correct"] and ft_pred["correct"]:
            both_correct.append(filename)
        elif not base_pred["correct"] and not ft_pred["correct"]:
            details = {
                "filename": filename,
                "actual_label": base_pred["actual_label"],
                "base_prediction": base_pred["predicted_label"],
                "finetuned_prediction": ft_pred["predicted_label"]
            }
            both_incorrect.append(details)
        elif not base_pred["correct"] and ft_pred["correct"]:
            details = {
                "filename": filename,
                "actual_label": base_pred["actual_label"],
                "base_prediction": base_pred["predicted_label"],
                "finetuned_prediction": ft_pred["predicted_label"]
            }
            finetuned_fixed.append(details)
        elif base_pred["correct"] and not ft_pred["correct"]:
            details = {
                "filename": filename,
                "actual_label": base_pred["actual_label"],
                "base_prediction": base_pred["predicted_label"],
                "finetuned_prediction": ft_pred["predicted_label"]
            }
            finetuned_broke.append(details)

    # --- 4. Print the Results ---
    print("\n" + "="*50)
    print(f"Model Comparison Summary: {dataset_name}")
    print("="*50)
    print(f"Total common images compared: {len(common_filenames)}")
    print(f"Both models CORRECT:       {len(both_correct):,}")
    print(f"Both models INCORRECT:     {len(both_incorrect):,}")
    print(f"IMPROVED by finetuning:    {len(finetuned_fixed):,}")
    print(f"REGRESSIONS after finetuning: {len(finetuned_broke):,}")
    print("="*50 + "\n")

    # --- 5. Print Samples ---
    print(f"--- Sample Predictions for {dataset_name} ---\n")
    sampled_classes_correct = set()
    for img in both_correct:
        class_name = img.split('_')[0]
        if class_name not in sampled_classes_correct:
            print(f"Both Correct: {img}")
            sampled_classes_correct.add(class_name)
        if len(sampled_classes_correct) >= 5:
            break

    print("-" * 20)

    sampled_classes_incorrect = set()
    for pred in both_incorrect:
        class_name = pred['filename'].split('_')[0]
        if class_name not in sampled_classes_incorrect:
            print(f"Both Incorrect: {pred}")
            sampled_classes_incorrect.add(class_name)
        if len(sampled_classes_incorrect) >= 5:
            break
    print("\n")


In [58]:
from dataset import GorillaReIDDataset
base_predictions_path = "/sc/home/iven.schlegelmilch/bachelor_thesis_code/base_predictions.json"
finetuned_predictions_path = "/sc/home/iven.schlegelmilch/bachelor_thesis_code/finetuned_predictions.json"

dataset_dir = "/workspaces/vast-gorilla/gorillawatch/data3/stratified_open_split_NEW/spac23+24-body_face-squared-deduplicated"

split_name = "test"
split_dir = os.path.join(dataset_dir, split_name)
split_files = [f for f in os.listdir(split_dir) if f.lower().endswith((".jpg", ".png"))]

split_dataset = GorillaReIDDataset(
    image_dir=split_dir,
    filenames=split_files,
    transform=None,
    base_mask_dir="/workspaces/vast-gorilla/gorillawatch/iven_thesis/masks",
    mask_transform=None,
    k=5,
)

valid_indices_for_ce_knn = split_dataset.images_for_ce_knn

# 2. Convert these indices into a SET of FULL filenames.
#    This set will have filenames with extensions like '.png'.
valid_filenames_for_ce = {split_dataset.filenames[i] for i in valid_indices_for_ce_knn}

print(f"\nGenerated a set of {len(valid_filenames_for_ce)} valid full filenames for CE KNN analysis.")

# 3. Run the analysis, passing this single set.
analyze_and_print_comparison(
    dataset_name="Standard Dataset (Strict CE KNN)",
    base_predictions_path=base_predictions_path,
    finetuned_predictions_path=finetuned_predictions_path,
    valid_full_filenames_set=valid_filenames_for_ce # Pass the set here
)

# Define paths for the second (zoo) dataset
base_zoo_predictions_path = "/sc/home/iven.schlegelmilch/bachelor_thesis_code/base_zoo_predictions.json"
finetuned_zoo_predictions_path = "/sc/home/iven.schlegelmilch/bachelor_thesis_code/finetuned_zoo_predictions.json"

analyze_and_print_comparison(
    dataset_name="Zoo Dataset",
    base_predictions_path=base_zoo_predictions_path,
    finetuned_predictions_path=finetuned_zoo_predictions_path,
    valid_full_filenames_set=None # No filter for this one
)


Filtering images for KNN evaluation with k=5...
Found 4767 / 5008 images suitable for Cross-Encounter KNN.
Found 5008 / 5008 images suitable for Standard KNN.
Total unique labels: 16
Total unique encounters: 93
CE KNN unique labels: 13, unique encounters: 91

Generated a set of 4767 valid full filenames for CE KNN analysis.

############################################################
### Analyzing Dataset: Standard Dataset (Strict CE KNN)
############################################################

Successfully loaded base predictions from /sc/home/iven.schlegelmilch/bachelor_thesis_code/base_predictions.json
Successfully loaded finetuned predictions from /sc/home/iven.schlegelmilch/bachelor_thesis_code/finetuned_predictions.json
Created base map with 4767 entries and finetuned map with 4767 entries.
Filtering enabled. Query images and ALL neighbors must be in the valid set of 4767 files.

Model Comparison Summary: Standard Dataset (Strict CE KNN)
Total common images compared: 4767
B